In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Locate the repository root so paths work regardless of the working directory.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
# r/litigi is an Italian subreddit: analyse hours in local time.
TZ = "Europe/Rome"
from subreddit_lens import get_parent_author, load_comments, local_hour, preprocess

In [ ]:
filename = DATA_DIR / "litigi_comments.parquet"
litigi = load_comments(filename)
litigi['created_dt'] = pd.to_datetime(litigi['created_utc'], unit='s', utc=True).dt.tz_convert(TZ)
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))
litigi = get_parent_author(litigi)
litigi.head()

In [ ]:
df_pivot = pd.pivot_table(
    litigi,
    index='author',
    columns="sentiment",
    aggfunc='size'
)

In [ ]:
df_pivot=df_pivot.fillna(0)
df_pivot=df_pivot.sort_values('positive',ascending=False)
df_pivot['positive_ratio']=df_pivot['positive']/(df_pivot['negative']+df_pivot['neutral'])
df_pivot['p/n']=df_pivot['positive']/(df_pivot['negative'])

In [ ]:
df_pivot

In [ ]:
litigi[(litigi['num_words']==1) & (litigi['body_preprocessed']!='[deleted]')][['body_preprocessed']]

In [ ]:
df_pivot.to_csv(OUTPUT_DIR / 'litigi_sentiment.csv')

In [ ]:
litigi[(litigi['body_preprocessed']!='[deleted]') &(litigi['body_preprocessed']!='[removed]') ]['num_words'].hist(bins=range(75),grid=False)

In [ ]:
sns.kdeplot(litigi[ (litigi['num_words']<100) &(litigi['num_words']>1) ]['num_words'])
plt.show()

In [ ]:
author_list=['tommyrugby','sda_express','SpiegoLeDiscussioni','PHEELZ']

In [ ]:
df_filtered = litigi[litigi['author'].isin(author_list)]

In [ ]:
for author in author_list:
    plt.hist(df_filtered[df_filtered['author'] == author]['num_words'], alpha=0.5, label=author,bins=range(75))
plt.legend(loc='upper right')
plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.show()

In [ ]:

histograms = []
for author in author_list:
    author_hist, _ = np.histogram(df_filtered[df_filtered['author'] == author]['num_words'],bins=range(50))
    histograms.append(author_hist)

# Plot histograms side by side
plt.figure(figsize=(19, 10))
width = 0.2
x = np.arange(len(histograms[0]))

for i, author_hist in enumerate(histograms):
    plt.bar(x + i*width, author_hist, width=width, alpha=0.9, label=author_list[i])

plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()


In [ ]:
date=pd.to_datetime(litigi['created_utc'], unit='s', utc=True).dt.tz_convert(TZ).dt.date
date

In [ ]:
plt.hist(date,bins=50,alpha=0.5)

plt.show()

In [ ]:
time=local_hour(litigi['created_utc'], tz=TZ).astype(int)
time

In [ ]:
plt.hist(time,alpha=0.9,rwidth=0.7,bins=24)

plt.show()

In [ ]:
histograms = []
for author in author_list:
    author_hist, _ = np.histogram(local_hour(df_filtered[df_filtered['author'] == author]['created_utc'], tz=TZ).astype(int),bins=24,density=True)
    histograms.append(author_hist)

# Plot histograms side by side
plt.figure(figsize=(19, 10))
width = 0.2
x = np.arange(len(histograms[0]))

for i, author_hist in enumerate(histograms):
    plt.bar(x + i*width, author_hist, width=width, alpha=0.9, label=author_list[i])

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
df_filtered = litigi[litigi['author'].isin(author_list)]

In [ ]:
plt.figure(figsize=(19, 10))

for author in author_list:
    sns.kdeplot(local_hour(df_filtered[df_filtered['author'] == author]['created_utc'], tz=TZ).astype(int),label=author)
    #histograms.append(author_hist)

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
plt.figure(figsize=(19, 10))

sns.kdeplot(data=df_filtered,x=local_hour(df_filtered['created_utc'], tz=TZ).astype(int),label=author,hue='author', common_norm=False)
plt.show()